# Escape Velocity — the four analyses (whitepaper §10)

> **Snapshot:** the site is a dated July 2026 piece — Padraig's position in C7.5 is on the
> record as of 11 July 2026 — so the price series is fetched live but cut at the **10 July
> close**, the last session before that date. Change `SNAPSHOT_END` below to extend it.

This notebook does five things, in order:

1. **Fetch** SPCX daily prices via `yfinance` (with a clearly-labelled stub fallback).
2. **Analysis 1 — money left on the table:** what the +19.2% pop cost the company.
3. **Analysis 2 — drawdown & volatility:** the peak-to-trough give-back, and how wild the ride was vs QQQ.
4. **Analysis 3 — the implied-growth / multiple calculation:** what today's price *requires* of the business.
5. **Analysis 4 — the float-supply model:** tradable shares through December 2026.

Each analysis writes its output JSON into `/data` (with `source` and `as_of` fields) —
those files are exactly what the website reads. Nothing on the site is typed in by hand.

In [1]:
# Setup: paths and the two constants every analysis shares.
# All figures here are whitepaper-sourced (§2, §5): $135 IPO price, 555,555,555 shares.
import json
import math
from datetime import date, timedelta
from pathlib import Path

DATA = Path("..") / "data"          # run from /analysis; JSONs land next to the site's data
SNAPSHOT_END = "2026-07-10"         # last close the site covers (the C7.5 byline is 11 July)
AS_OF = SNAPSHOT_END               # every JSON is "as of" the snapshot, not the day it ran
RETRIEVED = date.today().isoformat()

IPO_PRICE = 135.0
SHARES_SOLD = 555_555_555           # the base offering (greenshoe excluded, as in §10.1)
IPO_VALUATION_BN = 1770.0           # $1.77tn at $135 — the offering share base
EBITDA_2025_BN = 6.6                # 2025 adjusted EBITDA (§4)

## 1 · Fetch SPCX daily data

We ask Yahoo Finance for the full SPCX history since the 12 June 2026 debut, plus QQQ
over the same window (needed for the volatility comparison in Analysis 2).

**If the fetch fails** (no network, or the ticker is unavailable in this environment),
we fall back to a stub series hand-shaped to the whitepaper's *reported* anchors —
open \$150, day-one close \$160.95, peak \$225.64, ~\$149–150 by 7–8 July — and mark the
output `"is_stub": true` so every downstream chart admits it.

In [2]:
def fetch_live():
    import yfinance as yf
    # end= is exclusive in yfinance, so ask for the day after the snapshot
    end = (date.fromisoformat(SNAPSHOT_END) + timedelta(days=1)).isoformat()
    spcx = yf.download("SPCX", start="2026-06-12", end=end, auto_adjust=False, progress=False)
    if spcx.empty:
        raise ValueError("yfinance returned no rows for SPCX")
    qqq = yf.download("QQQ", start="2026-06-12", end=end, auto_adjust=False, progress=False)
    # newer yfinance returns (field, ticker) column pairs even for one ticker
    spcx.columns = spcx.columns.get_level_values(0)
    qqq.columns = qqq.columns.get_level_values(0)
    rows = [
        {"date": idx.strftime("%Y-%m-%d"),
         "open": round(float(r["Open"]), 2), "high": round(float(r["High"]), 2),
         "low": round(float(r["Low"]), 2), "close": round(float(r["Close"]), 2),
         "volume": int(r["Volume"])}
        for idx, r in spcx.iterrows()
    ]
    qqq_closes = [float(c) for c in qqq["Close"]]
    return rows, qqq_closes, False


def fetch_stub():
    # Anchor closes from the whitepaper; the values BETWEEN anchors are invented
    # shape only, which is why everything downstream carries the stub label.
    closes = [
        ("2026-06-12", 160.95, 500_000_000), ("2026-06-15", 171.20, 210_000_000),
        ("2026-06-16", 183.50, 190_000_000), ("2026-06-17", 196.40, 175_000_000),
        ("2026-06-18", 207.10, 160_000_000), ("2026-06-22", 216.30, 140_000_000),
        ("2026-06-23", 222.80, 130_000_000), ("2026-06-24", 225.64, 150_000_000),
        ("2026-06-25", 214.90, 145_000_000), ("2026-06-26", 203.30, 130_000_000),
        ("2026-06-29", 191.75, 120_000_000), ("2026-06-30", 182.40, 110_000_000),
        ("2026-07-01", 172.10, 100_000_000), ("2026-07-02", 164.55, 90_000_000),
        ("2026-07-06", 157.20, 85_000_000),  ("2026-07-07", 150.10, 120_000_000),
        ("2026-07-08", 149.50, 80_000_000),
    ]
    rows, prev = [], 150.0   # day-one open
    for i, (d, close, vol) in enumerate(closes):
        o = 150.0 if i == 0 else prev
        rows.append({"date": d, "open": round(o, 2), "high": round(max(o, close) * 1.012, 2),
                     "low": round(min(o, close) * 0.988, 2), "close": close, "volume": vol})
        prev = close
    return rows, None, True


try:
    rows, qqq_closes, is_stub = fetch_live()
    source = f"Yahoo Finance via yfinance, 12 Jun – {SNAPSHOT_END}, retrieved {RETRIEVED}"
except Exception as err:
    print("Live fetch failed (%s) — writing STUB data instead." % err)
    rows, qqq_closes, is_stub = fetch_stub()
    source = ("STUB — hand-shaped to reported anchors in the-offering-whitepaper.md §5–6. "
              "Run this notebook with live access to replace.")

json.dump(
    {"source": source, "as_of": AS_OF, "is_stub": is_stub, "ticker": "SPCX",
     "ipo_price": IPO_PRICE, "rows": rows},
    open(DATA / "spcx_daily.json", "w"), indent=2)

closes = [r["close"] for r in rows]
print(f"{len(rows)} trading days · last close ${closes[-1]} · stub={is_stub}")

19 trading days · last close $145.3 · stub=False


## 2 · Analysis 1 — money left on the table

"Money left on the table" is what the pop cost the seller: the company sold shares at
\$135 that the market was willing to pay \$160.95 for by the close. Multiply the gap by
the shares sold and you get the money that went to allocation recipients instead of to
SpaceX:

$$ 555{,}555{,}555 \times (\$160.95 - \$135) \approx \$14.4\text{bn} $$

That sounds enormous — but as a *percentage* (19.2%) it is small next to famous debuts
(Airbnb +113%, Snowflake +112%, Alibaba +38%). This is the whitepaper's genuinely good
interview point: whatever else you think of a take-it-or-leave-it price, **it priced
efficiently**. The output ranks SPCX inside the pop table the site shows in C6.

In [3]:
day_one_close = 160.95   # reported (§5) — not taken from the fetched series, so this
                         # analysis is identical on stub and live data
left_bn = SHARES_SOLD * (day_one_close - IPO_PRICE) / 1e9

money = {
    "source": "Computed: 555,555,555 shares x ($160.95 - $135). Whitepaper §10.1.",
    "as_of": AS_OF,
    "shares": SHARES_SOLD,
    "ipo_price": IPO_PRICE,
    "day_one_close": day_one_close,
    "left_on_table_bn": round(left_bn, 1),
    "underpricing_pct": round((day_one_close / IPO_PRICE - 1) * 100, 1),
    "verdict": "Small by mega-IPO pop standards — evidence the fixed price worked.",
}
json.dump(money, open(DATA / "money_left.json", "w"), indent=2)
money

{'source': 'Computed: 555,555,555 shares x ($160.95 - $135). Whitepaper §10.1.',
 'as_of': '2026-07-10',
 'shares': 555555555,
 'ipo_price': 135.0,
 'day_one_close': 160.95,
 'left_on_table_bn': 14.4,
 'underpricing_pct': 19.2,
 'verdict': 'Small by mega-IPO pop standards — evidence the fixed price worked.'}

## 3 · Analysis 2 — drawdown & volatility

Two questions about the aftermath: **how far did it fall** from the hype peak
(peak-to-trough drawdown), and **how wild is the ride day to day** (the standard
deviation of daily returns, annualised by √252 because variance grows with time)?
The whitepaper reports a ~32% correction from the \$225.64 peak; here we compute it
from the actual series. QQQ over the same window is the sanity check — a thin-float
mega-cap should be far more volatile than the index it just joined.

In [4]:
peak = max(closes)
peak_i = closes.index(peak)
trough = min(closes[peak_i:])
drawdown_pct = (trough / peak - 1) * 100

def annualised_vol(series):
    rets = [series[i] / series[i - 1] - 1 for i in range(1, len(series))]
    mean = sum(rets) / len(rets)
    daily = math.sqrt(sum((r - mean) ** 2 for r in rets) / (len(rets) - 1))
    return daily * 100, daily * math.sqrt(252) * 100

daily_vol, annual_vol = annualised_vol(closes)
qqq_annual = round(annualised_vol(qqq_closes)[1], 1) if qqq_closes else None

dd = {
    "source": ("Computed from spcx_daily.json. " +
               ("STUB series — run with live data. " if is_stub else "") +
               "Computed on daily closes; the whitepaper's ~-32% is measured from the $225.64 intraday high."),
    "as_of": AS_OF,
    "is_stub": is_stub,
    "peak": peak,
    "peak_date": rows[peak_i]["date"],
    "trough": trough,
    "drawdown_pct": round(drawdown_pct, 1),
    "reported_drawdown_pct": -32,
    "daily_vol_pct": round(daily_vol, 2),
    "annualised_vol_pct": round(annual_vol, 1),
    "qqq_annualised_vol_pct": qqq_annual,   # None until run with live access
}
json.dump(dd, open(DATA / "drawdown.json", "w"), indent=2)
dd

{'source': "Computed from spcx_daily.json. Computed on daily closes; the whitepaper's ~-32% is measured from the $225.64 intraday high.",
 'as_of': '2026-07-10',
 'is_stub': False,
 'peak': 211.39,
 'peak_date': '2026-06-16',
 'trough': 145.3,
 'drawdown_pct': -31.3,
 'reported_drawdown_pct': -32,
 'daily_vol_pct': 7.9,
 'annualised_vol_pct': 125.5,
 'qqq_annualised_vol_pct': 29.1}

## 4 · Analysis 3 — the implied-growth / multiple calculation

SPCX listed at ~94× trailing adjusted EBITDA; trillion-dollar peers trade at 15–40×.
Rather than just gasp at the gap, **invert it**: assume the multiple must normalise by
2030, and ask what EBITDA the business then needs to be worth today's price.

$$ \text{required EBITDA}_{2030} = \frac{EV}{\text{exit multiple}} \qquad
   \text{CAGR} = \left(\frac{\text{required}}{\$6.6\text{bn}}\right)^{1/5} - 1 $$

We compute a scenario grid on two EV bases: the \$1.77tn offering valuation, and that
valuation scaled to the latest close (the same maths behind the C7.5 slider). Note the
share-count caveat: \$1.77tn (offering valuation) and ~\$2.1tn (day-one market cap) use
different bases — one of the §16 discrepancies the site declares rather than hides.

In [5]:
YEARS = 5   # 2025 EBITDA -> 2030
last_close = closes[-1]
current_ev_bn = IPO_VALUATION_BN * last_close / IPO_PRICE

def scenario_grid(ev_bn):
    grid = []
    for multiple in [15, 20, 25, 30, 40, 50]:
        required = ev_bn / multiple
        cagr = (required / EBITDA_2025_BN) ** (1 / YEARS) - 1
        grid.append({"exit_multiple": multiple,
                     "required_ebitda_2030_bn": round(required, 1),
                     "implied_ebitda_cagr_pct": round(cagr * 100, 1)})
    return grid

implied = {
    "source": ("Computed per whitepaper §10.3: required EBITDA = EV / exit multiple; "
               "CAGR from $6.6bn 2025 adjusted EBITDA over 5 years. EV uses the $1.77tn "
               "offering-valuation share base (see §16 discrepancy register)."),
    "as_of": AS_OF,
    "ebitda_2025_bn": EBITDA_2025_BN,
    "horizon_years": YEARS,
    "bases": {
        "at_ipo_price": {"price": IPO_PRICE, "ev_bn": IPO_VALUATION_BN,
                         "scenarios": scenario_grid(IPO_VALUATION_BN)},
        "at_last_price": {"price": last_close, "ev_bn": round(current_ev_bn, 0),
                          "scenarios": scenario_grid(current_ev_bn)},
    },
}
json.dump(implied, open(DATA / "implied_growth.json", "w"), indent=2)

for s in implied["bases"]["at_last_price"]["scenarios"]:
    print(f"exit {s['exit_multiple']:>2}x -> needs ${s['required_ebitda_2030_bn']:>6}bn "
          f"2030 EBITDA -> {s['implied_ebitda_cagr_pct']}%/yr growth")

exit 15x -> needs $ 127.0bn 2030 EBITDA -> 80.7%/yr growth
exit 20x -> needs $  95.3bn 2030 EBITDA -> 70.6%/yr growth
exit 25x -> needs $  76.2bn 2030 EBITDA -> 63.1%/yr growth
exit 30x -> needs $  63.5bn 2030 EBITDA -> 57.3%/yr growth
exit 40x -> needs $  47.6bn 2030 EBITDA -> 48.5%/yr growth
exit 50x -> needs $  38.1bn 2030 EBITDA -> 42.0%/yr growth


## 5 · Analysis 4 — the float-supply model

Only ~4% of the equity floated at IPO — a deliberately thin float that amplified the
run to +67%. But lockups release in waves: ~20% of insider holdings after the 6 August
earnings, an acceleration trigger at \$175.50 before 3 August worth ~456m extra shares,
and a projected ~40% tradable by December (Musk himself locked 366 days).

The endpoints (4%, ~40%) and the wave mechanics are reported; **the monthly path
between them is an estimate**, and both the JSON and the site's chart say so. The
visual argument this feeds (C6): *supply is the gravity acting on the hype.*

In [6]:
supply = {
    "source": ("ESTIMATES hand-built from reported lockup terms (whitepaper §6, §9): 4% float "
               "at IPO; first wave ~20% of insider holdings after 6 Aug earnings; $175.50/3 Aug "
               "trigger would add ~456m shares; ~40% tradable by December; Musk locked 366 days. "
               "Intermediate points are interpolations — label as estimates wherever shown."),
    "as_of": AS_OF,
    "is_estimate": True,
    "trigger": {"price": 175.5, "deadline": "2026-08-03", "shares_m": 456},
    "points": [
        {"month": "JUN", "date": "2026-06-12", "float_pct": 4,  "note": "IPO — deliberately thin"},
        {"month": "JUL", "date": "2026-07-31", "float_pct": 4,  "note": "pre-earnings lockup holds"},
        {"month": "AUG", "date": "2026-08-31", "float_pct": 12, "note": "first wave after 6 Aug earnings (est.)"},
        {"month": "SEP", "date": "2026-09-30", "float_pct": 18, "note": "est."},
        {"month": "OCT", "date": "2026-10-31", "float_pct": 25, "note": "est."},
        {"month": "NOV", "date": "2026-11-30", "float_pct": 32, "note": "est."},
        {"month": "DEC", "date": "2026-12-31", "float_pct": 40, "note": "reported projection ~40%"},
    ],
}
json.dump(supply, open(DATA / "supply.json", "w"), indent=2)

# Quick visual: float bars with the price series overlaid (§10.4)
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar([p["month"] for p in supply["points"]],
        [p["float_pct"] for p in supply["points"]], color="#F0F0EC", edgecolor="#8A8D96")
ax1.set_ylabel("tradable float, % of shares")
ax2 = ax1.twinx()
ax2.plot(range(len(closes)), closes, color="#16171B", linewidth=1.5)
ax2.axhline(supply["trigger"]["price"], color="#E5484D", linestyle="--", linewidth=1)
ax2.set_ylabel("SPCX close, $ (dashed: $175.50 trigger)")
ax2.set_xticks([])
ax1.set_title("Supply is the gravity acting on the hype" + (" (STUB price data)" if is_stub else ""))
plt.tight_layout()
plt.show()

/var/folders/qj/kn8p2qd963757_1rkjzv4qm40000gp/T/ipykernel_22776/4223151209.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
### Done

Files written to `/data`: `spcx_daily.json`, `money_left.json`, `drawdown.json`,
`implied_growth.json`, `supply.json`. Reload the dev server and the site picks them up —
no code changes needed. If any output above says **STUB**, the live fetch failed — re-run when you have
live access.